Proyecto de RAG usando langchain

en base a tutorial: Keerti Purswani - Educosys.

In [1]:
!pip install langchain_community langchainhub chromadb langchain langchain_openai

In [80]:
# Hacemos Web Scraping - Google Supplier Code of Conduct
from langchain_community.document_loaders import WebBaseLoader
url = "https://about.google/intl/es_ALL/company-info/supplier-code-of-conduct/"
loader = WebBaseLoader(url)
docs = loader.load()
print(docs)

[Document(metadata={'source': 'https://about.google/intl/es_ALL/company-info/supplier-code-of-conduct/', 'title': 'Código de Conducta para Proveedores - Google - About Google', 'description': 'Conoce el compromiso de Google para asegurarse de que las condiciones laborales en sus operaciones y en sus cadenas de suministro sean seguras.', 'language': 'es-419'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nCódigo de Conducta para Proveedores - Google - About Google\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n\n\n \n\n\n \n\n \n\n \n\n\n\n\n\n\n\n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n\n \n\n\n \n\n \n\n \n\n \n\n \n\n\n\n\n\n\n \n\n \n\n \n\n \n\n \n\n \n\n\n\n\n\n \n\n\n\n\n\n\n\n\n \n\n \n\n \n\n \n\n\n\n\n \n\n \n\n\n\n\n\n\n\n \n\n \n\n\n\n\n \n\n\n\n\n\n\n \n\n\n\n\n\n\n \n\n \n

In [81]:
print(type(docs))
print(type(docs[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [83]:
#Imprimimos un pedazo del texto:
print(docs[0].page_content[600:3000])

1. Introducción
Google se compromete a tratar a todos los trabajadores con respeto y dignidad, a velar por unas condiciones laborales saludables y seguras, y a llevar a cabo sus operaciones de forma ética y responsable con el medioambiente. Google espera lo mismo de sus proveedores, distribuidores, partners de contratación de personal, contratistas, subcontratistas y proveedores subcontratados ("Proveedores"). El presente Código de Conducta para Proveedores ("Código") establece las expectativas sobre cómo deben operar las empresas con las que hacemos negocios. Estas expectativas se derivan de estándares y marcos reconocidos internacionalmente, incluidos los Principios Rectores sobre las Empresas y los Derechos Humanos de las Naciones Unidas y los Derechos Humanos. El Código se aplica a todos los Proveedores de productos y servicios de Alphabet, Google y sus empresas subsidiarias ("Google").
Los Proveedores deben implementar y monitorizar programas diseñados para lograr que el Código se

In [84]:
# Sustituimos Google por una empresa ficticia
# para comprobar posteriormente el grounding del RAG

for doc in docs:
    doc.page_content = doc.page_content.replace(
        "Google",
        "FalconiCorp2026"
    )

print(docs[0].page_content[630:3000])

Introducción
FalconiCorp2026 se compromete a tratar a todos los trabajadores con respeto y dignidad, a velar por unas condiciones laborales saludables y seguras, y a llevar a cabo sus operaciones de forma ética y responsable con el medioambiente. FalconiCorp2026 espera lo mismo de sus proveedores, distribuidores, partners de contratación de personal, contratistas, subcontratistas y proveedores subcontratados ("Proveedores"). El presente Código de Conducta para Proveedores ("Código") establece las expectativas sobre cómo deben operar las empresas con las que hacemos negocios. Estas expectativas se derivan de estándares y marcos reconocidos internacionalmente, incluidos los Principios Rectores sobre las Empresas y los Derechos Humanos de las Naciones Unidas y los Derechos Humanos. El Código se aplica a todos los Proveedores de productos y servicios de Alphabet, FalconiCorp2026 y sus empresas subsidiarias ("FalconiCorp2026").
Los Proveedores deben implementar y monitorizar programas diseñ

In [85]:
#Vamos a dividir la documentación en chunks, usamos herramienta de langchain

#Nota: langchain.text_splitters ha cambiado a langchain_text_splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 200,
    length_function = len,
)
splits = text_splitter.split_documents(docs)


In [87]:
#inspeccionamos los chunks:
#imprimiendo 2 splits seguidos podemos ver el overlap de información
print(len(splits))
print()
print(splits[5].page_content)
print()
print(splits[6].page_content)

46

1.3 Certificaciones. Los Proveedores deben usar certificaciones fiables de terceros, incluidos los materiales, los procesos de producción y los productos, cuando corresponda. Si FalconiCorp2026 solicita certificaciones adicionales, como la ISO 50001, la 14001 o la 45001, los Proveedores deben esforzarse de buena fe para obtener dichas certificaciones de manera oportuna.
1.4 Responsabilidad. A pesar de que intentemos trabajar con los Proveedores para mejorar las condiciones, puede que suspendamos o resolvamos nuestra relación con cualquier Proveedor que no cumpla las responsabilidades del Código.
2. Prácticas laborales justas y derechos humanos

2. Prácticas laborales justas y derechos humanos
FalconiCorp2026 espera que sus Proveedores compartan su compromiso con la defensa de los derechos humanos de los trabajadores, su trato digno y respetuoso, y un lugar de trabajo libre de acoso, discriminación y represalias. Esto se aplica a todos los trabajadores, incluidos los temporales, mig

In [9]:
#EMBEDDINGS E INDEXACIÓN EN CHROMA
#Convertimos los chunks en embeddings y los almacenamos
#en una base de datos vectorial

#El tutorial original propone usar embedding de OpenAI mediante API
#from langchain_openai import OpenAIEmbeddings
#from langchain_chroma import Chroma

#Nosotros usaremos embedding local de Hugginface:

!pip install -q langchain-huggingface sentence-transformers



In [10]:
#Continua :

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

print("Vector store creado correctamente")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector store creado correctamente


In [11]:
#el total:
print(vectorstore._collection.count())

46


In [12]:
#los id:
print(vectorstore._collection.get())

{'ids': ['814062f5-6e29-448d-b054-adbd8de32b75', '10766ff1-1805-431a-b147-33792e5a44da', '30feec7d-3c52-4e73-9ccc-4dc96efe6999', 'c1cf239a-2c96-4ac8-8ccb-ecce41ce785d', '2667d856-9c7f-4d0b-8254-61f45689be2f', '3776a61d-accb-4c82-9c77-4d37a3480d85', 'deafc56b-e951-4b41-9145-c0fb45564f21', '1d7d604f-6e5f-458f-876d-85b369fd1a3d', '07b2e7ba-9a69-4446-bb4e-3491c00186e2', 'a19f0acf-e475-453f-a89f-eace647a9459', '92a6ba6a-871e-4501-a0ac-84b80d5e1191', '53215660-35fa-441d-9727-67f0751a1709', 'cbf6e809-9990-4057-8ca2-bfd93f62507b', 'c5db5ff5-56fc-4efe-a53a-574bf74857ae', 'e8d2df72-de4d-4b2c-ba5d-db433fcaa8e8', 'de71827b-07a0-49bc-883d-a255373c5275', '842511db-4d84-4e73-a9d6-7fd7fbcbb5ee', '0e6c89c9-7af4-47c5-ba16-938dcba58082', '56185f4f-b691-4e79-8f01-12a69a22d9ce', '8b1ac967-2a23-4e39-90d9-53ecbf3b95b4', '76659044-623b-4d26-8b7f-fab8c7715059', 'b12fea7f-e907-4771-b9a6-d6ae473ae445', 'e6dae2f8-0848-4176-a610-8a3b322bef3f', '7d37bc44-5f7c-4e77-bf78-f463484550b0', 'c1081d3b-f6ce-4956-84fe-5eab40

In [13]:
#imprimiendo una coleccion:
print("\nCollection 1 -", vectorstore._collection.get(ids=["b65e94fa-24ab-4d33-abf9-dbd377908c58"]))


Collection 1 - {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [14]:
#Comenzamos con el RAG PIPELINE

In [31]:
#necesitamos que el vector store actue como el retriever
#y lo configuramos para que por cada pregunta me recupere los 3 chunks mas similares
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [39]:
#probamos el retriever:

question = "¿Qué dice FalconiCorp2026 sobre las horas de trabajo?"

retrieved_docs = retriever.invoke(question)

print("Chunks recuperados:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- CHUNK {i + 1} ---")
    print(doc.page_content)

Chunks recuperados: 3

--- CHUNK 1 ---
1. Introducción
FalconiCorp2026 se compromete a tratar a todos los trabajadores con respeto y dignidad, a velar por unas condiciones laborales saludables y seguras, y a llevar a cabo sus operaciones de forma ética y responsable con el medioambiente. FalconiCorp2026 espera lo mismo de sus proveedores, distribuidores, partners de contratación de personal, contratistas, subcontratistas y proveedores subcontratados ("Proveedores"). El presente Código de Conducta para Proveedores ("Código") establece las expectativas sobre cómo deben operar las empresas con las que hacemos negocios. Estas expectativas se derivan de estándares y marcos reconocidos internacionalmente, incluidos los Principios Rectores sobre las Empresas y los Derechos Humanos de las Naciones Unidas y los Derechos Humanos. El Código se aplica a todos los Proveedores de productos y servicios de Alphabet, FalconiCorp2026 y sus empresas subsidiarias ("FalconiCorp2026").

--- CHUNK 2 ---
2.4 

In [45]:
#vemos el score del retriever:
results = vectorstore.similarity_search_with_score(
    question,
    k=3
)

for doc, score in results:
    print(doc.page_content)
    print("Score:", score)
    print("-----")

1. Introducción
FalconiCorp2026 se compromete a tratar a todos los trabajadores con respeto y dignidad, a velar por unas condiciones laborales saludables y seguras, y a llevar a cabo sus operaciones de forma ética y responsable con el medioambiente. FalconiCorp2026 espera lo mismo de sus proveedores, distribuidores, partners de contratación de personal, contratistas, subcontratistas y proveedores subcontratados ("Proveedores"). El presente Código de Conducta para Proveedores ("Código") establece las expectativas sobre cómo deben operar las empresas con las que hacemos negocios. Estas expectativas se derivan de estándares y marcos reconocidos internacionalmente, incluidos los Principios Rectores sobre las Empresas y los Derechos Humanos de las Naciones Unidas y los Derechos Humanos. El Código se aplica a todos los Proveedores de productos y servicios de Alphabet, FalconiCorp2026 y sus empresas subsidiarias ("FalconiCorp2026").
Score: 13.82375431060791
-----
2.4 Horario laboral. No se tr

In [32]:
#Creando el prompt

#En vez de construir el prompt manualmente,
#usaremos una plantilla de langchain
#Definimos las instrucciones y los placeholders que se rellenarán
#dinámicamente con el contexto recuperado y la pregunta del usuario.

from langchain_core.prompts import ChatPromptTemplate

template = """
Eres un asistente que responde preguntas utilizando únicamente
la información proporcionada en el contexto.

Si la respuesta no aparece en el contexto, responde:
"No tengo suficiente información en los documentos."

Contexto:
{context}

Pregunta:
{question}

Respuesta:
"""

prompt = ChatPromptTemplate.from_template(template)

In [33]:
# Usaremos una LLM LOCAL y ligera - QWEN 2.5 0.5B

"""
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"
)
"""

'\nfrom transformers import pipeline\n\nllm = pipeline(\n    "text-generation",\n    model="Qwen/Qwen2.5-0.5B-Instruct",\n    device_map="auto"\n)\n'

In [54]:
#Pero hay un problema:

# El tutorial usa from langchain_openai import ChatOpenAI
#llm = ChatOpenAI(), es un objeto LLM de LangChain

#Y nosotros estamos usando QWEN que es de hugginface (un pipeline de hugginface Transformers)
#entones tenemos que "envolver" a Qwen para que sea equivalente:

#Langchain tiene un wrapper para pipelines de hugginface, entonces:

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# 1. Pipeline de Transformers
hf_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    max_new_tokens=200
)

# 2. Lo envolvemos como LLM de LangChain
llm = HuggingFacePipeline(
    pipeline=hf_pipeline
)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [55]:
#Usamos RunnablePassthrough para que cada vez que entre una nueva pregunta,
#esa entrada se pasa directamente al campo question.

from langchain_core.runnables import RunnablePassthrough

#Tambien importamos StrOutputParser, toma la salida del LLM y retorna solo texto
from langchain_core.output_parsers import StrOutputParser


In [56]:
#Juntamos los textos
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [57]:
#Construimos la cadena RAG CHAIN
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [58]:
#probando:
rag_chain.invoke("¿Qué dice FalconiCorp2026 sobre las horas de trabajo?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nEres un asistente que responde preguntas utilizando únicamente\nla información proporcionada en el contexto.\n\nSi la respuesta no aparece en el contexto, responde:\n"No tengo suficiente información en los documentos."\n\nContexto:\n1. Introducción\nFalconiCorp2026 se compromete a tratar a todos los trabajadores con respeto y dignidad, a velar por unas condiciones laborales saludables y seguras, y a llevar a cabo sus operaciones de forma ética y responsable con el medioambiente. FalconiCorp2026 espera lo mismo de sus proveedores, distribuidores, partners de contratación de personal, contratistas, subcontratistas y proveedores subcontratados ("Proveedores"). El presente Código de Conducta para Proveedores ("Código") establece las expectativas sobre cómo deben operar las empresas con las que hacemos negocios. Estas expectativas se derivan de estándares y marcos reconocidos internacionalmente, incluidos los Principios Rectores sobre las Empresas y los Derechos Humanos de las Nac

In [59]:
#Nos da un resultado sin la respuesta esperada,
#vamos a ajustar los parametros del modelo

# Configuración corregida para CPU
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    max_new_tokens=80,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Convertimos los Document recuperados a texto antes de insertarlos en el prompt.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG listo. Dispositivo:", hf_pipeline.model.device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

RAG listo. Dispositivo: cpu


In [60]:
#Volvemos a Probar el modelo local pero con datos ajustados:
#probando:
rag_chain.invoke("¿Qué dice FalconiCorp2026 sobre las horas de trabajo?")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'"El presente Código de Conducta para Proveedores (\'Código\') establece las expectativas sobre cómo deben operar las empresas con las que hacemos negocios. Estas expectativas se derivan de estándares y marcos reconocidos internacionalmente, incluidos los Principios Rectores sobre las Empresas y los Derechos Humanos de las Naciones Unidas y los'

In [61]:
#Comclusión: el LLM usado no nos ofrece la calidad que esperamos
#Podríamos pensar en mejorar la etapa de retrieval,
#Sin embargo habría un límite de todas maneras en el razonamiento del modelo
#Por lo que por cuestiones prácticas:
#Usaremos otro LLM, por medio de API
#Optamos por uno gratuito,
#pasaremos de  Qwen 0.5B local a GPT-OSS 20B

!pip install -qU langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.9 MB/s eta 0:00:00


In [62]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GroqTestKey")

In [66]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=200
)

print(type(llm))

<class 'langchain_groq.chat_models.ChatGroq'>


In [67]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [70]:
llm.invoke("Qué dia es hoy?")

AIMessage(content='Hoy es **28 de septiembre de 2026**.', additional_kwargs={'reasoning_content': 'The user asks in Spanish: "Qué día es hoy?" They want the current date. The system says current date is 2026-09-28. So answer: 28 de septiembre de 2026.'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 76, 'total_tokens': 142, 'completion_time': 0.073099479, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.005322598, 'prompt_tokens_details': None, 'queue_time': 0.917134127, 'total_time': 0.078422077}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e98f-044d-7472-bed0-851a3564c81e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 66, 'total_tokens': 142, 'output_token_details': {'reasoning': 45}})

In [69]:
#Volvemos a probar:
rag_chain.invoke("¿Qué dice FalconiCorp2026 sobre las horas de trabajo?")


'FalconiCorp2026 establece que los trabajadores no deben trabajar más de **60 horas a la semana** (incluidas las horas extra) o el máximo establecido por la ley, el que sea inferior. Se permiten excepciones cuando:\n\n1. El convenio colectivo lo permita.  \n2. El trabajador sea un empleado profesional.  \n3. El trabajo se realice en respuesta a una urgencia o situación puntual que no se haya podido prever'

In [72]:
#Ahora si :)
#Tenemos un RAG con una generación aceptable.
